# 14 · Datos del Estado: INGEMMET, INDECI (SINPAD), CENEPRED, ANA y ANIN

Qué publican estas entidades, qué se pudo obtener sin cuenta y qué mejora para el catálogo de huaicos y para las quebradas con nombre.

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Ante una emergencia llama al 115 (INDECI).

**Fuentes y atribución.** INGEMMET: GEOCATMIN (https://geocatmin.ingemmet.gob.pe), usado bajo su *Licencia para uso no comercial* (se cita a INGEMMET como fuente original). INDECI: portal público del SINPAD. CENEPRED: SIGRID (servicios ArcGIS FEN 2027 sin clave). ANA y ANIN: notas públicas de las entidades. Todo es información pública del Estado; la descarga es opt-in y moderada (una petición cada 1,5 s, sin credenciales).

In [ ]:
import json
from pathlib import Path

import pandas as pd

from illapa_research import fuentes_estado as FE
from illapa_research.catalogo import cargar_todos
from illapa_research.quebradas import cargar_referencias

RAIZ = Path("..") if Path("../catalogo").exists() else Path(".")
CAT = RAIZ / "catalogo"
DATOS = RAIZ / "datos"
cuencas = json.loads((CAT / "cuencas.geojson").read_text(encoding="utf-8"))
indice = FE.IndiceCuencas(cuencas)
refs = cargar_referencias(CAT / "quebradas_referencias.csv")
HAY_ING = (DATOS / "ingemmet" / "peligros_geologicos.geojson").exists()
HAY_SINPAD = (DATOS / "indeci" / "sinpad_zona.csv").exists()
print("INGEMMET en datos/:", HAY_ING, "| SINPAD en datos/:", HAY_SINPAD)
if not (HAY_ING and HAY_SINPAD):
    print("Para bajarlos (moderado, sin cuenta): uv run python -m illapa_research.fuentes_estado")

## 1. Inventario por entidad

| Entidad | Qué hay | Acceso sin cuenta | Útil para |
|---|---|---|---|
| INGEMMET (GEOCATMIN) | Inventario de peligros geológicos (1 433 puntos en la zona), zonas críticas (108), «Perú Alerta» (zonas críticas, activación de quebrada) | Sí: `MapServer` de ArcGIS, con el certificado del sistema | Nombres y puntos de quebradas; no trae la fecha del evento (la fecha es del estudio) |
| INDECI (SINPAD) | Lista pública de emergencias, 136 305 registros, desde 2019 | Sí: API JSON anónima (`ubigeo`, `fechaInicio`, `fechaFin`) | Fechas oficiales de huaico por distrito (2019-2026) |
| CENEPRED (SIGRID) | Documentos y mapas; servicios `FEN/ER_NINO2027_BD` y `sectores/COEN_FEN_2027` | Solo esos servicios; el resto pide token y no se usó | Puntos críticos 2027 con nombre, población y peligro; riesgo por distrito |
| ANA | 755 puntos críticos y 12 barreras dinámicas (notas de prensa) | Lista nominal solo en prensa; la capa de puntos críticos 2027 está en SIGRID | Quebradas con barrera |
| ANIN | Cartera de 13 quebradas del Rímac (Andina, 10-09-2026) | Notas de prensa; el sitio web exige otra vía | Lista oficial de quebradas priorizadas |

## 2. Puntos con nombre de INGEMMET frente a nuestras cuencas

Para cada quebrada con nombre, los puntos del inventario cuyo nombre la menciona («Qda.», «Quebrada» o «Huayco») y la cuenca donde caen. Es una validación independiente de la ubicación que hicimos con las fajas marginales de la ANA y la ordenanza de Lima.

In [ ]:
import re


def cargar(k):
    return FE.puntos_a_tabla(json.loads((DATOS / "ingemmet" / f"{k}.geojson").read_text(encoding="utf-8")))

if HAY_ING:
    capas = {"peligros_geologicos": ["NM_SECTOR", "DENOMINACION"], "zonas_criticas": ["PARAJE"], "alerta_zonas_criticas": ["PARAJE"]}
    partes = []
    for k, cols in capas.items():
        df = cargar(k)
        df["nombre_lugar"] = df[cols].fillna("").astype(str).agg(" / ".join, axis=1).str.strip(" /")
        r = [indice.de_punto(x, y) for x, y in zip(df.lon, df.lat, strict=True)]
        df["cuenca"], df["dentro"], df["dist_m"] = zip(*r, strict=True)
        df["capa"] = k
        partes.append(df[["capa", "nombre_lugar", "lon", "lat", "cuenca", "dentro", "dist_m"]])
    T = pd.concat(partes, ignore_index=True)
    T["nn"] = T["nombre_lugar"].map(FE.normalizar)
    nombres = {f["properties"]["nombre"].replace("Quebrada ", ""): f["properties"]["slug"]
               for f in cuencas["features"] if f["properties"]["ubicacion"] == "referencia"}
    filas = []
    for r in refs:
        sl = nombres.get(r["nombre"])
        if not sl:
            continue
        toks = {FE.normalizar(a) for a in (r["nombre"], *r["alias"]) if len(a) > 3}
        m = T[T["nn"].map(lambda s, toks=toks: any(t in s for t in toks) and bool(re.search(r"\b(qda|quebrada|huayco)\b", s)))]
        filas.append({"quebrada": r["nombre"], "puntos": len(m), "en_nuestra_cuenca": int((m["cuenca"] == sl).sum()),
                      "en_otra_cuenca": int(((m["cuenca"] != sl) & m["cuenca"].notna()).sum()),
                      "fuera_de_toda_cuenca": int(m["cuenca"].isna().sum())})
    V = pd.DataFrame(filas)
    display(V)
    print("TOTAL", V[["puntos", "en_nuestra_cuenca", "en_otra_cuenca", "fuera_de_toda_cuenca"]].sum().to_dict())

## 3. SINPAD frente a nuestro catálogo (2019-2026)

Las emergencias con peligro principal «HUAICO» de los distritos de estudio: ¿ya estaban en el catálogo?

In [ ]:
if HAY_SINPAD:
    d = pd.read_csv(DATOS / "indeci" / "sinpad_zona.csv")
    cat = {e.fecha for e in cargar_todos(solo_dia_exacto=True)}
    e = FE.emergencias_de_lluvia(d)
    h = e[e["peligroPrincipal"].str.upper().str.contains("HUAICO") & e["distrito"].isin(FE.DISTRITOS_ZONA)].copy()
    h["en_catalogo"] = pd.to_datetime(h["fecha"]).dt.date.isin(cat)
    display(h[["fecha", "hora_local", "distrito", "nivel", "codigoSinpad", "en_catalogo"]])
    print(h["en_catalogo"].value_counts().to_dict(), "(las False se agregaron en catalogo/eventos_sinpad_2026.csv)")

## 4. Listas oficiales de quebradas priorizadas frente a nuestro ranking

In [ ]:
rk = pd.read_csv(CAT / "ranking_bayes_quebradas_nino.csv")
cmp_ = FE.comparar_priorizadas(refs, rk)
display(cmp_[["entidad", "quebrada_oficial", "medida", "slug", "rango_mediano", "p_top5"]])

## Lo que NO hay público

- **Fecha del evento en el inventario de INGEMMET:** la columna `FECHA` es la del estudio (2003, 2008, 2021...), no la del huaico.
- **SINPAD antes de 2019:** la lista pública solo trae la versión 2.1; para 1981-2018 hace falta pedir la base histórica a INDECI (Ley 27806).
- **Capas de CENEPRED (peligro, riesgo, elementos expuestos) y servicios de ANA/ANIN:** piden token o son aplicaciones sin API pública.
- **El inventario de movimientos en masa del SENAMHI (IMMS):** ver el issue 1 del repositorio.